# Modeling

Trains Linear Regression and Random Forest on the same train/test split and compares them.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
import os
os.chdir(project_root)

In [2]:
import pandas as pd

from src.data.load_data import load_config, load_raw_data
from src.data.preprocess import clean_data
from src.features.feature_engineering import split_data, split_X_y
from src.models.evaluate import evaluate_model
from src.models.train import save_model, train_model

config = load_config()
clean = clean_data(load_raw_data(config))
train_df, test_df = split_data(clean, config)
X_train, y_train = split_X_y(train_df)
X_test, y_test = split_X_y(test_df)

## Train and evaluate both models

Each model is trained on the same `X_train`/`y_train`, saved to `models/`, and scored on the same held-out `X_test`/`y_test`. Every run also gets appended to `reports/model_runs.csv`.

In [3]:
results = {}
for model_name in ["linear_regression", "random_forest"]:
    model = train_model(model_name, X_train, y_train, config)
    save_model(model, model_name, config)
    results[model_name] = evaluate_model(model_name, X_test, y_test, config)

pd.DataFrame(results).T

,rmse,mae,r2
linear_regression,29556.458489,23442.091802,0.469497
random_forest,29568.132726,23659.609284,0.469078


The two models are statistically indistinguishable - RMSE differs by about 12 KSh out of ~29,560 (well within noise), and R² differs in the third decimal place. Random Forest's ability to model non-linear relationships isn't finding anything to exploit beyond what a linear combination of `Bedrooms`, `Bathrooms`, and `Estate` already captures.

Given the tie, **Linear Regression is the project's designated primary model** (see `config.yaml: output.primary_model`) - with equal accuracy, the simpler and more interpretable model is the better choice; Random Forest's added complexity buy nothing here.

An R² of ~0.47 means these three features explain under half the variance in price - a real limitation of the feature set (no amenities, building condition, or agency data available), not a modeling defect. See the README's Limitations section for more.

## Run history

Every run's hyperparameters and metrics are logged here - useful for comparing across experiments over time, not just the two runs from this session.

In [4]:
pd.read_csv("reports/model_runs.csv").tail()

,timestamp,model_name,hyperparameters,rmse,mae,r2
0,2026-08-20T17:45:28,linear_regression,"{'copy_X': True, 'fit_intercept': True, 'n_job...",29556.46,23442.09,0.4695
1,2026-08-20T17:45:28,random_forest,"{'bootstrap': True, 'ccp_alpha': 0.0, 'criteri...",29568.13,23659.61,0.4691
2,2026-08-20T19:00:36,linear_regression,"{'copy_X': True, 'fit_intercept': True, 'n_job...",29556.46,23442.09,0.4695
3,2026-08-20T19:00:37,random_forest,"{'bootstrap': True, 'ccp_alpha': 0.0, 'criteri...",29568.13,23659.61,0.4691
